[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mdgordillob/aca_aci_collab/blob/main/notebooks/aca_opt_benchmark.ipynb)

# aca_opt pipeline benchmark

Clones the **source ACI-CO pipeline repo** (`aca_indice_climatico_opt`, not this lightweight companion repo) and runs three of its real pipeline stages -- Stage 1 (merge/resample raw grib to daily), Stage 2 (baseline percentiles), Stage 3 (temperature anomalies) -- directly, as imported Python functions, against one year of raw ERA5 fetched from Drive. Each stage is timed so you can see how this pipeline performs on Colab's hardware and extrapolate to a full 1961--2024 run.

**This is a timing/mechanics smoke test, not a scientifically valid percentile calculation** -- it deliberately uses a single year of data so it finishes quickly. Stage 2 normally runs once over the full 30-year 1961--1990 baseline; see the note in that section for how to extrapolate.

The pipeline scripts needed no changes for this -- `unir_archivos.py`, `calcular_percentil_temperatura.py`, and `calcular_anomalias_temperatura.py` already expose plain, importable functions behind their `__main__` CLI wrappers.

In [ ]:
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
IN_COLAB

## 1. Get the code

This clones the **pipeline** repo (`aca_indice_climatico_opt`), not `aca_aci_collab` -- the two are separate: this notebook lives in the lightweight companion repo, but it benchmarks the real pipeline.

In [ ]:
import sys, os

REPO_ROOT = "/content/aca_indice_climatico_opt"

if IN_COLAB:
    !git clone --depth 1 https://github.com/mdgordillob/aca_indice_climatico_opt.git {REPO_ROOT}
else:
    REPO_ROOT = os.path.abspath("../../aca_indice_climatico_opt-main")  # adjust if running locally

sys.path.insert(0, os.path.join(REPO_ROOT, "src", "scripts"))
sys.path.insert(0, os.path.join(REPO_ROOT, "src", "utils"))
sys.path.insert(0, os.path.join(REPO_ROOT, "src"))  # for drive_sync

## 2. Install dependencies

Colab's base image doesn't have `cfgrib`'s
system-level `eccodes` library, or `rioxarray`/`geopandas` (used for the
shapefile clip step). This repo's own environment already has these, so
this cell is a no-op locally.

In [ ]:
if IN_COLAB:
    !apt-get -qq install -y libeccodes-dev > /dev/null
    !pip install -q cfgrib eccodes rioxarray geopandas

## 3. Mount Drive and fetch one year of raw ERA5

`DRIVE_ROOT` is confirmed to be the right path ("My Drive/Indice Climatico Actuarial/2. Datos") -- edit it only if your own Drive differs. This folder is shared with specific collaborators, not "anyone with the link" (confirmed from the Drive UI's access panel), which is why anonymous fetching isn't available -- this uses `drive.mount()` instead, which authenticates as whoever is logged into this Colab session.

`era5/` turned out to have its own sub-structure (`completos/`, `union/`, `Combined/`, `percentiles_nc/`, ...) rather than flat `.grib` files directly, and which subfolder holds `era5_tmp_<year>.grib` isn't confirmed yet. The cell below tries the likely candidates in order and tells you which one worked (or didn't) -- if none do, run `explore_drive_structure.ipynb` and add the right name to `ERA5_GRIB_CANDIDATES`.

Only **one year** of temperature grib is copied locally -- enough for a timing smoke test without a slow full-archive transfer. Change `BENCHMARK_YEAR` to any year you confirm is present.

In [ ]:
import shutil

DRIVE_ROOT = "/content/drive/MyDrive/Indice Climatico Actuarial/2. Datos"
BENCHMARK_YEAR = 1985
ERA5_GRIB_CANDIDATES = ["completos", "union", ""]  # "" = era5/ itself

raw_dir = os.path.join(REPO_ROOT, "data", "raw", "era5")
os.makedirs(raw_dir, exist_ok=True)

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    grib_name = f"era5_tmp_{BENCHMARK_YEAR}.grib"
    src = None
    for candidate in ERA5_GRIB_CANDIDATES:
        probe = os.path.join(DRIVE_ROOT, "era5", candidate, grib_name)
        if os.path.exists(probe):
            src = probe
            print(f"found {grib_name!r} under era5/{candidate!r}")
            break
    if src is None:
        raise FileNotFoundError(
            f"Couldn't find {grib_name} under era5/ in any of {ERA5_GRIB_CANDIDATES}. "
            "Run explore_drive_structure.ipynb, find the right subfolder, and add "
            "it to ERA5_GRIB_CANDIDATES above."
        )

    dst = os.path.join(raw_dir, grib_name)
    shutil.copy(src, dst)
    print(f"copied {dst} ({os.path.getsize(dst) / 1e6:.0f} MB)")

    import drive_sync
    drive_sync.sync(DRIVE_ROOT, REPO_ROOT, only=["shapefiles"])
else:
    print("Not in Colab -- assuming data/raw/era5 and data/shapefiles are already populated locally.")

## 4. Stage 1 -- merge/resample raw grib to daily

`unir_archivos.process_yearly_data_tmp` decodes the hourly GRIB file with `cfgrib` and resamples to daily max/min. This is the expensive, per-year, embarrassingly-parallel step.

In [ ]:
import time
import unir_archivos

stage1_out = os.path.join(REPO_ROOT, "data", "processed", "_benchmark_daily_tmp")
grib_path = os.path.join(raw_dir, f"era5_tmp_{BENCHMARK_YEAR}.grib")

t0 = time.perf_counter()
unir_archivos.process_yearly_data_tmp(grib_path, BENCHMARK_YEAR, "t2m", stage1_out)
unir_archivos.merge_yearly_files(
    stage1_out,
    os.path.join(stage1_out, "era5_daily_combined_tmp_benchmark.nc"),
    "t2m",
)
stage1_time = time.perf_counter() - t0
print(f"Stage 1 (merge/resample, {BENCHMARK_YEAR} only): {stage1_time:.1f}s")

## 5. Stage 2 -- baseline percentiles

`calcular_percentil_temperatura.calcular_percentiles` normally runs once over the full 1961--1990 baseline (30 years) already merged by Stage 1. Here it runs against the single merged year from above instead, purely to time the computation -- the *shape* of the work (groupby-month quantiles over however many years are in the input) is the same; only the year count differs. **Multiply this time by ~30 for a rough estimate of the real baseline computation.**

In [ ]:
import calcular_percentil_temperatura as percentil_tmp

merged_file = os.path.join(stage1_out, "era5_daily_combined_tmp_benchmark.nc")

t0 = time.perf_counter()
estadisticas = percentil_tmp.calcular_percentiles(merged_file)
percentil_tmp.guardar_percentiles(
    estadisticas,
    os.path.join(stage1_out, "percentiles_benchmark.nc"),
    stage1_out,
    guardar_csv=False,
)
stage2_time = time.perf_counter() - t0
print(f"Stage 2 (percentiles, {BENCHMARK_YEAR} only -- not a real 30-year baseline): {stage2_time:.1f}s")

## 6. Stage 3 -- temperature anomalies

`calcular_anomalias_temperatura.procesar_anomalias_temperatura` reads raw grib directly (not Stage 1's output) plus the Stage 2 percentile file, and writes one NetCDF per year-month plus a combined CSV -- the same shape of output `aci_lib` reads as `anomalias_colombia/anomalies_temperature` in the other notebook.

In [ ]:
import calcular_anomalias_temperatura as anomalias_tmp

stage3_out = os.path.join(REPO_ROOT, "data", "processed", "_benchmark_anomalias")
os.makedirs(stage3_out, exist_ok=True)
shapefile_path = os.path.join(REPO_ROOT, "data", "shapefiles", "colombia_4326.shp")

t0 = time.perf_counter()
anomalias_tmp.procesar_anomalias_temperatura(
    archivo_percentiles=os.path.join(stage1_out, "percentiles_benchmark.nc"),
    archivo_comparar_location=raw_dir,
    output_csv_path=os.path.join(stage3_out, "anomalies_temperature_benchmark.csv"),
    shapefile_path=shapefile_path if os.path.exists(shapefile_path) else None,
    output_netcdf=stage3_out,
    use_multiprocessing=False,  # single year -- not worth spinning up a pool
)
stage3_time = time.perf_counter() - t0
print(f"Stage 3 (anomalies, {BENCHMARK_YEAR} only): {stage3_time:.1f}s")

## 7. Summary

In [ ]:
import pandas as pd

summary = pd.DataFrame([
    {"stage": "1. merge/resample", "years_in_this_run": 1, "seconds": stage1_time},
    {"stage": "2. baseline percentiles", "years_in_this_run": 1, "seconds": stage2_time},
    {"stage": "3. anomalies", "years_in_this_run": 1, "seconds": stage3_time},
])
summary["est_seconds_full_1961_2024"] = summary["seconds"] * 64
summary

Stages 1 and 3 scale roughly linearly with the number of years processed (each year is decoded/resampled independently), so the `est_seconds_full_1961_2024` column is a straightforward extrapolation. Stage 2 in the real pipeline runs once, over a fixed 30-year window, not once per year -- its `est_seconds_full_1961_2024` figure isn't meaningful; see the note in Section 5 instead.